In [ ]:
# Script for Agent loop getting groceries, finding prices and applying discount to total bill.

!pip install anthropic


import anthropic
import json
key = '<key>'

In [ ]:
client = anthropic.Anthropic(api_key=key)

# Define the tools

def orders(orderNumber):

  db = {
      "123" : [
          {"name" : "laptop", "price" : 100},
          {'name': 'cereal', 'price': 10}
      ],
      "456" : [
          {'name' : 'guitar', 'price' : 500},
          {'name' : 'cup', 'price' : 2}
      ]
  }

  return {'items' : db.get(str(orderNumber)) }

def sum_prices(prices):
  return sum(prices)

def apply_discount(price, discount_val):
  return price - (price * discount_val)

def run_tools(name, inputs):

  if name == 'orders':
    return orders(inputs['orderNumber'])
  elif name == 'sum_prices':
    return sum_prices(inputs['prices'])
  elif name == 'apply_discount':
    return apply_discount(inputs['price'], inputs['discount_val'])
  else:
    return ValueError('Tool not found.')

tools = [
    {
        'name' : 'orders',
        'description': 'the function which returns all the open orders',
        'input_schema': {
            'type' : 'object',
            'properties' : {
                'orderNumber' : {'type': 'string'}
            },
            'required': ['orderNumber']
        }
    },
    {
        'name' : 'sum_prices',
        'description': 'Summing up the prices of the order items',
        'input_schema' : {
            'type' : 'object',
            'properties' : {
                'prices' : {
                    'type': 'array',
                    'items' : {'type': 'number'}
                    }
            },
            'required' : ['prices']
        }
    },
    {
        'name' : 'apply_discount',
        'description' : 'the function for applying the discount',
        'input_schema' : {
            'type' : 'object',
            'properties' : {
                'price' : {'type' : 'number'},
                'discount_val' : {'type' : 'number'}
            },
            'required' : ['discount_val', 'price']
        }

    }
]

messages = [
        {
            'role' : 'user',
            'content' : 'What is the discount price of the order 123 if discount applied in 10%?'
        }
    ]

response = client.messages.create(
    model = 'claude-opus-4-6',
    max_tokens = 1024,
    tools = tools,
    messages = messages
)

# Agentic Loop
while response.stop_reason == 'tool_use':

  for block in response.content:
    if block.type == 'tool_use':

        tool_results = []

        try:

          result = run_tools(block.name, block.input)

          tool_results.append({
              'type': 'tool_result',
              'tool_use_id' : block.id,
              'content': json.dumps(result)
          })

        # Raise exception if there is an issue with tool_use
        except Exception as ex:

          tool_results.append({
              'type': 'tool_result',
              'tool_use_id': block.id,
              'content' : str(ex),
              'is_error' : True
          })

        messages.append({
            "role" : 'assistant',
            'content' : response.content
        })
        messages.append({
            "role" : 'user',
            'content' : tool_results
        })

        response = client.messages.create(
            model = 'claude-opus-4-6',
            max_tokens = 1024,
            tools = tools,
            messages = messages
        )

final_message = next(block for block in response.content if block.type == 'text')
print(final_message)



TextBlock(citations=None, text="Here's the summary for **Order #123**:\n\n| Item    | Price  |\n|---------|--------|\n| Laptop  | $100   |\n| Cereal  | $10    |\n| **Total** | **$110** |\n| **Discount (10%)** | **-$11** |\n| **Final Price** | **$99** |\n\nAfter applying a **10% discount** on the total of **$110**, the discounted price for Order 123 is **$99.00**. 🎉", type='text')
